### Init

In [1]:
import sys
sys.path.append('../src/')

In [2]:
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS, GLOBAL_COMBINED_STATS
import hydra
from hydra.utils import instantiate
from pathlib import Path
import os
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import logging

from utils.data_utils import (
    get_wet_mask,
    get_train_test_ranges,
    gen_data_in_test,
    gen_data_out_test,
    data_CNN_Lateral,
    data_CNN_Dynamic,
    gen_data_025_lateral,
    gen_data_global_new,
)
from utils.eval_utils import (
    generate_model_rollout,
    compute_mean,
    compute_var,
    compute_corrs_area,
    compute_rmse,
    compute_corrs,
    compute_KE,
    compute_time_spec,
    compute_ACC,
    compute_nino34,
    compute_amo,
    gen_KE_spectrum,
    gen_KE,
    gen_KE_range,
    gen_value_range,
    gen_enstrophy_spectrum,
    gen_enstrophy,
    compute_corrs_single,
    compute_ACC_single,
    compute_RMSE_single,
    compute_mean_single,
)
from utils.subgrid_utils import get_area_tensor
from utils.climate_utils import compute_laplacian_wet
from utils.plot_utils import (
    plot_short_time_stats,
    plot_long_time_stats,
    plot_map,
    plot_error_map,
    plot_both_error_map,
    plot_metrics_KE_spectrum,
    plot_metrics_KE,
    plot_metrics_enstrophy_spectrum,
    plot_metrics_entrophy,
    plot_metrics_corr,
    plot_metrics_rmse,
    plot_metrics_acc,
    plot_metrics_mean,
    plot_metrics_pdf,
    get_initial_snapshot_fig,
    plot_region_based_metric,
    plot_diff_map,
)

import numpy as np
import torch
import xarray as xr
import copy

In [3]:
class Eval:
    def __init__(self, args, mode):
        # Getting input, extra input and output
        self.inputs = INPT_VARS[args.exp_num_in]
        self.extra_in = EXTRA_VARS[args.exp_num_extra]
        self.outputs = OUT_VARS[args.exp_num_out]

        self.str_in = "".join([i + "_" for i in self.inputs])
        self.str_ext = "".join([i + "_" for i in self.extra_in])
        self.str_out = "".join([i + "_" for i in self.outputs])

        print("inputs: " + self.str_in)
        print("extra inputs: " + self.str_ext)
        print("outputs: " + self.str_out)

        self.N_atm = len(self.extra_in)  # Number of atmosphere variables
        self.N_in = len(self.inputs)
        if args.lateral:
            self.N_extra = (
                self.N_atm + self.N_in
            )  # Number of atmosphere variables + Lateral boundary variables
        else:
            self.N_extra = self.N_atm  # Number of atmosphere variables
        self.N_out = len(self.outputs)

        self.num_in = int((args.hist + 1) * self.N_in + self.N_extra)

        print("Number of inputs: ", self.num_in)  # 3 (ocean speeds + ocean temp)(t) +
        # 3 (atm wind stresses + atm temp)(t) +
        # 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
        print("Number of outputs: ", self.N_out)  # 3

        # Post-fix strings
        self.str_train = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_out
            + "N_train_4000"
            + "_Lateral_Data_025_no_smooth"
        )
        self.str_save = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_"
            + args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )
        self.post_model_name = (
            "Train_" + args.train_region
            + "_Test_" + args.region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_in
            + "N_train_"
            + str(args.N_samples)
            + "_Lateral_Data_025_no_smooth"
        )
        self.post_pred_name = (
            args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )

        # Getting start and end indices of train and test
        s_train, e_train, e_test = get_train_test_ranges(
            args.N_samples, args.N_val, args.lag, args.hist, args.interval
        )

        # Saving data
        print("Getting inputs")
        if "global_1" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag)
        elif "global_2x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag, run_type ="2x")
        elif "global_4x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(self.inputs, self.extra_in, self.outputs, args.lag, run_type ="4x")
        else:
            raise NotImplementedError

        print("Calculating mask tensors")
        self.wet, self.wet_nan = get_wet_mask(inputs, "cpu")
        self.wet_bool = np.array(self.wet.cpu()).astype(bool)
        wet_lap = compute_laplacian_wet(self.wet_nan, 4) # hardcoded
        wet_lap = xr.where(wet_lap == 0, 1, np.nan)
        self.wet_lap = np.nan_to_num(wet_lap)
        print("Wet resolution:", self.wet.shape)

        self.time_vec = inputs[0].time.data

        self.time_test = self.time_vec[e_test : (e_test + args.lag * args.N_test)]

        print("Loading Train data")
        train_data = torch.load(
                    Path(args.data_dir) / "train_data_cnn_{0}.pt".format(self.str_train),
                    map_location=torch.device("cpu"),
                )
        # self.train_data = train_data
    
        if args.save_test_data:
            print("Saving data")
            data_in_test = gen_data_in_test(
                0, e_test, args.N_test, args.lag, args.hist, inputs, extra_in
            )
            data_out_test = gen_data_out_test(
                0, e_test, args.N_test, args.lag, args.hist, outputs
            )
            if "global" in args.region:
                norm_vals = train_data.norm_vals
                if "combined" in args.train_region:
                    assert len(norm_vals) == len(GLOBAL_COMBINED_STATS) and all(np.array_equal(norm_vals[k], GLOBAL_COMBINED_STATS[k]) for k in norm_vals)
                self.test_data = data_CNN_Dynamic(
                    data_in_test,
                    data_out_test,
                    self.wet.to(device="cpu"),
                    norm_vals,
                    device=args.device,
                )
                del train_data
            else:
                raise NotImplementedError()
            torch.save(
                self.test_data,
                Path(args.data_dir) / "test_{1}_data_cnn_{0}.pt".format(self.str_save, mode),
            )

        else:
            print("Loading test data")
            self.test_data = torch.load(
                Path(args.data_dir) / "test_{1}_data_cnn_{0}.pt".format(self.str_save, mode),
            )

        # Model
        print("Loading model " + args.network)
        if "swin" in args.network.lower():
            model = instantiate(
                args.swin,
                in_channels=self.num_in,
                output_channels=self.N_in,
                pretrain_img_size=[*self.test_data[0][0].shape[1:]],
                wet=self.wet.cuda()
            )
        elif "unet" in args.network.lower():
            model = instantiate(
                args.unet, wet=self.wet.cuda()
            )

        full_model_path = args.ckpt_path
        self.full_model_name = args.network + "_" + self.post_model_name
        self.output_channels = model.output_channels

        model = model.to(args.device)
        self.ckpt_path = args.ckpt_path
        self.model = model

        # Stats
        self.mean_out = self.test_data.norm_vals["m_out"]
        self.std_out = self.test_data.norm_vals["s_out"]
        self.mean_in = self.test_data.norm_vals["m_in"]
        self.std_in = self.test_data.norm_vals["s_in"]

        # clim
        self.clim = None
        if args.save_clim_data:
            print("Saving clim")
            clim = np.zeros((366, *self.wet.shape, 3))
            for i in range(self.N_out):
                clim[:, :, :, i] = (
                    outputs[i].groupby("time.dayofyear").mean("time").data
                )
            torch.save(
                clim,
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save),
            )

        else:
            print("Loading clim")
            clim = torch.load(
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save)
            )

        self.clim = clim

        # Getting area tensor
        print("Computing area tensor")
        self.grids = xr.open_dataset('/scratch/as15415/Data/CM2x_grids/Grid_New.nc').rename({"dx": "dxu", "dy": "dyu"})

        self.area = torch.from_numpy(self.grids["area_C"].to_numpy()).to(device="cpu")
        self.dx = self.grids["dxu"].to_numpy()
        self.dy = self.grids["dyu"].to_numpy()

        self.pred_model_path = Path(args.path_dir) / self.full_model_name
        if not os.path.isdir(self.pred_model_path):
            os.makedirs(self.pred_model_path)

        self.Nb = args.Nb
        self.hist = args.hist
        self.lag = args.lag
        self.N_test = args.N_test
        self.N_samples = args.N_samples
        self.output_dir = args.output_dir
        self.region = args.region
        self.steps = args.steps
        self.network = args.model_name_replace
        self.inputs = inputs

        self.pred_region = args.region
        self.pred_names = args.pred_names if args.pred_names else []
        self.pred_paths = args.pred_paths if args.pred_paths else []

        self.JUPYTER_MODE = False

    def send_data_to_cpu(self):
        self.test_data.set_device(device="cpu")

In [4]:
def generate_pred_lateral(e, args):
    print("Generation Pred begin...")
    ns = 4000
    for rand_ind, model_path in enumerate(args.ckpt_path):
        print("Random seed: ", rand_ind+1)
        e.model.load_state_dict(
            torch.load(model_path, map_location=torch.device('cuda'))
        )
        
        model_pred = generate_model_rollout(
            e.N_test,
            e.test_data,
            e.model,
            e.hist,
            e.N_in,
            e.N_extra,
            e.Nb,
            e.region,
        )

        print("data_gen")
        da = xr.DataArray(
            data=model_pred,
            dims=["time", "x", "y", "var"],
        )

        da.to_zarr(
            e.pred_model_path
            / (
                "Pred_lateral_Fast_Data_025_"
                + e.post_pred_name
                + "_rand_seed_"
                + str(rand_ind+1)
                + ".zarr"
            ),
            mode="w",
        )
        print(f"Model pred shape {model_pred.shape}")


### Choose model

In [5]:
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
import copy
from datetime import datetime
import os

# ConvNext
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args1 = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet",
#         "network=ConvNext UNet Train1Eval1-default",
#         "train_region=global_1",
#         "region=global_1",
#         "save_test_data=False", # Done Generation
#         "run_gen_pred=False", # Done Generation
#         "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed100/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed200/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args1.output_dir):
#     os.mkdir(args1.output_dir)

# Adam UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args1 = compose(config_name="exp/eval_adamunet_global", overrides=[
#         "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
#         "model_name_replace=Adam UNet",
#         "train_region=global_1",
#         "region=global_1",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Adam UNet Train1Eval1-default",
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_adamunet_global_1/adamunetseed/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed100/daam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed200/adam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args1.output_dir):
#     os.mkdir(args1.output_dir)

# Swin
with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
    args1 = compose(config_name="exp/eval_swin_global", overrides=[
        "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
        "model_name_replace=Swin",
        "train_region=global_1",
        "region=global_1",
        "run_gen_pred=True", # Multi-Seed Generation
        "network=Swin Train1Eval1-default",
        "swin.embed_dim=60",
        "exp/modules/blocks@swin.up_sampling_block=transposed_conv_upsample",
        "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_swintrans60_global_1/swintrans60/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed100/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed200/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
        "pred_names=null",
        "pred_paths=null"
    ])
if not os.path.exists(args1.output_dir):
    os.mkdir(args1.output_dir)

e1 = Eval(args1, mode='default')

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model Swin Train1Eval1-default


/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading clim
Computing area tensor


In [6]:
if args1.run_gen_pred:
    generate_pred_lateral(e1, args1)

Generation Pred begin...
Random seed:  1
[[-0.  0.  0.  0. -0.]
 [ 0. -0. -0. -0.  0.]
 [-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0. -0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
Random seed:  2
[[-0. -0. -0.  0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
Random seed:  3
[[-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]]
data_gen
Model pred shape (3000, 180, 360, 3)


In [7]:
# ConvNext minus
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args2 = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet |-1|",
#         "network=ConvNext UNet Train1Eval1-minus",
#         "save_test_data=False", # Done Generation
#         "run_gen_pred=False", # Done Generation
#         "train_region=global_1",
#         "region=global_1",
#         "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed100/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed200/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args2.output_dir):
#     os.mkdir(args2.output_dir)

# Adam UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args2 = compose(config_name="exp/eval_adamunet_global", overrides=[
#         "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
#         "model_name_replace=Adam UNet |-1|",
#         "train_region=global_1",
#         "region=global_1",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Adam UNet Train1Eval1-minus",
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_adamunet_global_1/adamunetseed/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed100/daam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed200/adam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args2.output_dir):
#     os.mkdir(args2.output_dir)

# Swin
with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
    args2 = compose(config_name="exp/eval_swin_global", overrides=[
        "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
        "model_name_replace=Swin |-1|",
        "train_region=global_1",
        "region=global_1",
        "run_gen_pred=True", # Multi-Seed Generation
        "network=Swin Train1Eval1-minus",
        "swin.embed_dim=60",
        "exp/modules/blocks@swin.up_sampling_block=transposed_conv_upsample",
        "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_swintrans60_global_1/swintrans60/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed100/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed200/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
        "pred_names=null",
        "pred_paths=null"
    ])
if not os.path.exists(args2.output_dir):
    os.mkdir(args2.output_dir)

e2 = Eval(args2, mode='minus')

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model Swin Train1Eval1-minus
Loading clim
Computing area tensor


In [8]:
if args2.run_gen_pred:
    generate_pred_lateral(e2, args2)

Generation Pred begin...
Random seed:  1
[[-0.  0.  0.  0. -0.]
 [ 0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0. -0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
Random seed:  2
[[-0. -0. -0.  0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
Random seed:  3
[[-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]]
data_gen
Model pred shape (3000, 180, 360, 3)


In [9]:
# ConvNext plus
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args3 = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet |+1|",
#         "network=ConvNext UNet Train1Eval1-plus",
#         "save_test_data=False", # Done Generation
#         "run_gen_pred=False", # Done Generation
#         "train_region=global_1",
#         "region=global_1",
#         "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed100/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed200/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args3.output_dir):
#     os.mkdir(args3.output_dir)

# Adam UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args3 = compose(config_name="exp/eval_adamunet_global", overrides=[
#         "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
#         "model_name_replace=Adam UNet |+1|",
#         "train_region=global_1",
#         "region=global_1",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Adam UNet Train1Eval1-plus",
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_adamunet_global_1/adamunetseed/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed100/daam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed200/adam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args3.output_dir):
#     os.mkdir(args3.output_dir)

# Swin
with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
    args3 = compose(config_name="exp/eval_swin_global", overrides=[
        "output_dir=./temp/{0}_perturbation".format(str(datetime.now())[:10]),
        "model_name_replace=Swin |+1|",
        "train_region=global_1",
        "region=global_1",
        "run_gen_pred=True", # Multi-Seed Generation
        "network=Swin Train1Eval1-plus",
        "swin.embed_dim=60",
        "exp/modules/blocks@swin.up_sampling_block=transposed_conv_upsample",
        "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_swintrans60_global_1/swintrans60/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed100/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed200/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
        "pred_names=null",
        "pred_paths=null"
    ])
if not os.path.exists(args3.output_dir):
    os.mkdir(args3.output_dir)
e3 = Eval(args3, mode='plus')

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs
Calculating mask tensors


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model Swin Train1Eval1-plus
Loading clim
Computing area tensor


In [10]:
if args3.run_gen_pred:
    generate_pred_lateral(e3, args3)

Generation Pred begin...
Random seed:  1
[[-0.  0.  0.  0. -0.]
 [ 0. -0. -0. -0.  0.]
 [-0. -0. -0. -0. -0.]
 [ 0.  0.  0.  0.  0.]
 [ 0. -0.  0.  0.  0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
Random seed:  2
[[-0. -0. -0.  0.  0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]]
data_gen
Model pred shape (3000, 180, 360, 3)
Random seed:  3
[[-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0.]]
data_gen
Model pred shape (3000, 180, 360, 3)


In [11]:
e1.send_data_to_cpu()
e2.send_data_to_cpu()
e3.send_data_to_cpu()

In [12]:
def load_seeded_data(Pred_path, e, current_net=False):
    if current_net:
        prefix =  "Pred_lateral_Fast_Data_025_" + e.post_pred_name + "_rand_seed_"
    else:
        prefix = ("Pred_lateral_Fast_Data_025_" 
                    + e.pred_region
                    + "_in_"
                    + e.str_in
                    + "ext_"
                    + "tau_u_tau_v_t_ref_"
                    + "N_samples_"
                    + str(e.N_samples)
                    + "_rand_seed_")

    model_preds = None

    for rand_seed in range(1, num_seeds + 1):
        print("Loading seed ", rand_seed)
        model_pred_net = (
            xr.open_zarr(
                Path(Pred_path) / (prefix + str(rand_seed) + ".zarr")
            )
        ).to_array().to_numpy().squeeze()
    
        if model_preds is None:
            model_preds = np.zeros((num_seeds,) + model_pred_net.shape)
    
        model_preds[rand_seed - 1] = model_pred_net
    
    return model_preds


def load_long_data(e):
    print("Load long data...")
    model_pred_net = load_seeded_data(e.pred_model_path, e, True)

    model_pred_saved_nets = []
    if len(e.pred_paths) > 0:
        for model_pred_path in e.pred_paths:
            model_pred_saved_nets.append(
                load_seeded_data(model_pred_path, e, False)
            )
        
    return model_pred_net, model_pred_saved_nets


In [13]:
def get_timeseries_temperature(e, model_pred_net, model_pred_saved_nets, start=0, N_eval=200, N_eval_ACC=100, long=False):
    ### Spatial matching metrics
    print("Getting Spatial matching stats...")
    T_test = np.array(
        e.test_data[:][1][:, 2] * e.std_out[2] + e.mean_out[2]
    )
    
    print("Getting Mean...")
    
    mean_T_net_list = []
    mean_T_true_list = []

    for i in range(num_seeds):
        mean_T_net, mean_T_true = compute_mean_single(
            N_eval,
            T_test,
            model_pred_net[i][:, :, :, 2],
            e.area,
            e.wet_bool,
        )
        mean_T_net_list.append(mean_T_net)
        mean_T_true_list.append(mean_T_true)
    
    mean_T_net = np.stack(mean_T_net_list)
    mean_T_true = np.stack(mean_T_true_list)
    
    mean_T_saved = []
    
    for model_pred_saved in model_pred_saved_nets:
        mean_T_i_list = []
        
        for i in range(num_seeds):
            mean_T_i, _ = compute_mean_single(
                N_eval,
                T_test,
                model_pred_saved[i][:, :, :, 2],
                e.area,
                e.wet_bool,
            )
            mean_T_i_list.append(mean_T_i)
        
        mean_T_saved.append(np.stack(mean_T_i_list))

    return mean_T_true, mean_T_saved + [mean_T_net]


In [14]:
num_seeds=3

In [15]:

model_pred_net, model_pred_saved_nets = load_long_data(e1)
mean_T_true1, mean_T_saved1 = get_timeseries_temperature(e1, model_pred_net, model_pred_saved_nets, start=1999, N_eval=2999, long=True)
model_pred_net, model_pred_saved_nets = load_long_data(e2)
mean_T_true2, mean_T_saved2 = get_timeseries_temperature(e2, model_pred_net, model_pred_saved_nets, start=1999, N_eval=2999, long=True)
model_pred_net, model_pred_saved_nets = load_long_data(e3)
mean_T_true3, mean_T_saved3 = get_timeseries_temperature(e3, model_pred_net, model_pred_saved_nets, start=1999, N_eval=2999, long=True)


Load long data...
Loading seed  1
Loading seed  2
Loading seed  3
Getting Spatial matching stats...
Getting Mean...
Load long data...
Loading seed  1
Loading seed  2
Loading seed  3
Getting Spatial matching stats...
Getting Mean...
Load long data...
Loading seed  1
Loading seed  2
Loading seed  3
Getting Spatial matching stats...
Getting Mean...


In [16]:
assert (mean_T_true1 == mean_T_true2).all()
assert (mean_T_true2 == mean_T_true3).all()
mean_T_true = mean_T_true1
mean_T_saved = [mean_T_saved1, mean_T_saved2, mean_T_saved3]

In [17]:
np.save(f'/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/temp/2024-05-24_perturbation/{args1.model_name_replace}_mean_T_true', mean_T_true)
np.save(f'/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/temp/2024-05-24_perturbation/{args1.model_name_replace}_mean_T_saved', mean_T_saved)